# Étape 7 — Manipulations : l'inférence OWL (symétrique, inverse, transitive)

Même logique **avant / après** qu'à l'étape 6, mais cette fois avec le raisonneur **OWL**, qui sait raisonner sur le comportement des propriétés.

> Prérequis : `pip install -r requirements.txt` (inclut `owlrl`). Exécute les cellules dans l'ordre.

In [ ]:
from rdflib import Graph

graphe = Graph()
graphe.parse("famille.ttl", format="turtle")
print(f"Triplets AVANT inference : {len(graphe)}\n")


def afficher(requete: str) -> None:
    resultats = graphe.query(requete)
    if not resultats:
        print("  (aucun resultat)")
        return
    for ligne in resultats:
        valeurs = {str(var): str(val) for var, val in ligne.asdict().items()}
        print(" ", valeurs)


# Trois requetes, une par type de propriete :
REQ_CONNAIT = """
PREFIX ex: <http://exemple.org/famille/>
SELECT ?qui ?connait WHERE { ?qui ex:connait ?connait . }
"""

REQ_PARENTS = """
PREFIX ex: <http://exemple.org/famille/>
SELECT ?enfant ?parent WHERE { ?enfant ex:aPourParent ?parent . }
"""

REQ_ANCETRES = """
PREFIX ex: <http://exemple.org/famille/>
SELECT ?personne ?ancetre WHERE { ?personne ex:aPourAncetre ?ancetre . }
"""

Triplets AVANT inference : 22



## 1. AVANT : seulement ce qu'on a écrit

- `connait` : seulement « Marie connaît Paul ».
- `aPourParent` : **rien** (on n'a écrit que `aPourEnfant`).
- `aPourAncetre` : seulement les deux maillons explicites (Lea→Marie, Marie→Jeanne), pas le raccourci Lea→Jeanne.

In [ ]:
print("--- connait (avant) ---")
afficher(REQ_CONNAIT)

print("\n--- aPourParent (avant) ---")
afficher(REQ_PARENTS)

print("\n--- aPourAncetre (avant) ---")
afficher(REQ_ANCETRES)

--- connait (avant) ---


  {'qui': 'http://exemple.org/famille/marie', 'connait': 'http://exemple.org/famille/paul'}

--- aPourParent (avant) ---
  (aucun resultat)

--- aPourAncetre (avant) ---
  {'personne': 'http://exemple.org/famille/lea', 'ancetre': 'http://exemple.org/famille/marie'}
  {'personne': 'http://exemple.org/famille/marie', 'ancetre': 'http://exemple.org/famille/jeanne'}


## 2. On lance le raisonneur OWL

`DeductiveClosure(OWLRL_Semantics)` applique les règles OWL (dont symétrie, inverse, transitivité) et enrichit le graphe sur place.

In [ ]:
from owlrl import DeductiveClosure, OWLRL_Semantics

avant = len(graphe)
DeductiveClosure(OWLRL_Semantics).expand(graphe)
apres = len(graphe)

print(f"Triplets avant : {avant}")
print(f"Triplets apres : {apres}")
print(f"Triplets deduits : {apres - avant}")

Triplets avant : 22
Triplets apres : 166
Triplets deduits : 144


## 3. APRÈS : les déductions apparaissent

On relance les mêmes requêtes :
- `connait` : maintenant « Paul connaît Marie » apparaît aussi (**symétrie**).
- `aPourParent` : « Lea a pour parent Marie » apparaît (**inverse** de aPourEnfant).
- `aPourAncetre` : « Lea a pour ancêtre Jeanne » apparaît (**transitivité**).

In [ ]:
print("--- connait (apres) ---")
afficher(REQ_CONNAIT)

print("\n--- aPourParent (apres) ---")
afficher(REQ_PARENTS)

print("\n--- aPourAncetre (apres) ---")
afficher(REQ_ANCETRES)

--- connait (apres) ---
  {'qui': 'http://exemple.org/famille/marie', 'connait': 'http://exemple.org/famille/paul'}
  {'qui': 'http://exemple.org/famille/paul', 'connait': 'http://exemple.org/famille/marie'}

--- aPourParent (apres) ---
  {'enfant': 'http://exemple.org/famille/lea', 'parent': 'http://exemple.org/famille/marie'}

--- aPourAncetre (apres) ---
  {'personne': 'http://exemple.org/famille/lea', 'ancetre': 'http://exemple.org/famille/marie'}
  {'personne': 'http://exemple.org/famille/marie', 'ancetre': 'http://exemple.org/famille/jeanne'}
  {'personne': 'http://exemple.org/famille/lea', 'ancetre': 'http://exemple.org/famille/jeanne'}


## À toi de jouer : une propriété symétrique « est marié avec »

**Exercice :** dans la cellule ci-dessous, on ajoute une nouvelle ontologie en mémoire avec une propriété symétrique `estMarieAvec`, on écrit un seul sens, on lance le raisonneur, et on vérifie que l'autre sens est déduit. Complète la déclaration manquante.

In [ ]:
exo = Graph()
exo.parse(data="""
@prefix ex:  <http://exemple.org/famille/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

# TODO : declare ex:estMarieAvec comme une owl:SymmetricProperty
ex:estMarieAvec a owl:ObjectProperty .

ex:paul ex:estMarieAvec ex:marie .
""", format="turtle")

DeductiveClosure(OWLRL_Semantics).expand(exo)

# On regarde si "Marie est mariee avec Paul" a ete deduit
for s, p, o in exo.triples((None, None, None)):
    if "estMarieAvec" in str(p):
        print(s, "estMarieAvec", o)

http://exemple.org/famille/paul estMarieAvec http://exemple.org/famille/marie


<details>
<summary>Afficher la solution</summary>

Remplace la ligne `ex:estMarieAvec a owl:ObjectProperty .` par :

```turtle
ex:estMarieAvec a owl:SymmetricProperty .
```

Avec `owl:ObjectProperty` seul, on ne voit qu'un sens (Paul → Marie). Avec `owl:SymmetricProperty`, le raisonneur ajoute aussi **Marie → Paul**.

</details>

---

Félicitations, tu as terminé le parcours ! Tu sais désormais : écrire des triplets, structurer en Turtle, interroger en SPARQL (simple puis avancé), modéliser un schéma en RDFS, et faire raisonner une ontologie avec RDFS puis OWL.